# Continuous Optimization in CANON

The module `NLPSLV` in `CANON` provides a wrapper to nonlinear optimization solvers, which can be either the commercial solver [SNOPT](https://ccom.ucsd.edu/~optimizers/solvers/snopt/) or the open-source solver [IPOPT](https://projects.coin-or.org/Ipopt) depending on compilation options. `CANON` leverages the DAG evaluation and automatic differentiation capability in `pyMC` to generates all the necessary function evaluations and derivatives internally. One can also leverage capability in `CRONOS` to embed ordinary differential equations (ODEs) into a DAG to enable dynamic optimization.

## Defining and Solving a Nonlinear Program (NLP)

Suppose we want to solve the continuous NLP:
$$\begin{align}
\max_{\bf x}\ & x_1+x_2 \\
\text{s.t.}\ \ & x_1\,x_2 \leq c \\
& 0 \leq x_1 \leq 6\\
& 0 \leq x_2 \leq 4\,.
\end{align}$$

We start by importing both the `PyMC`, `CRONOS` and `CANON` modules:

In [1]:
import pymc
import cronos
import canon

An environment `NLPSLV` is created and populated with the decision variables, cost and constraint expressions in the model:

In [2]:
# Define DAG
DAG = pymc.FFGraph()
X1 = pymc.FFVar(DAG,"X1")
X2 = pymc.FFVar(DAG,"X2")
C  = pymc.FFVar(DAG,"C")

In [3]:
# Define NLP
NLP = canon.NLPSLV()
NLP.set_dag( DAG )
NLP.add_parameter( [C] )
NLP.add_decision( [X1,X2], [0.,0.], [6.,4.] )
NLP.set_objective( NLP.MAX, X1+X2 )
NLP.add_constraint( NLP.LE, X1*X2-C )

Options can be modified as follows - these options can vary depending on the solver used:

In [4]:
NLP.options.FEASTOL   = 1e-8;
NLP.options.OPTIMTOL  = 1e-8;
NLP.options.GRADMETH  = NLP.options.BSYM;
NLP.options.DISPLEVEL = 1;
  
help( NLP.options )

Help on Options in module canon object:

class Options(pybind11_builtins.pybind11_object)
 |  Method resolution order:
 |      Options
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(...)
 |      __init__(*args, **kwargs)
 |      Overloaded function.
 |
 |      1. __init__(self: canon.NLPSLV.Options) -> None
 |
 |      2. __init__(self: canon.NLPSLV.Options, arg0: canon.NLPSLV.Options) -> None
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  DISPLEVEL
 |      Corresponds to 'Print file' in snOptA, which specifies the file name for the 'Summary file'. Displays to screen if an empty string is passed [Default: -]
 |
 |  FCTPREC
 |      Corresponds to 'Function precision' in snOptA, a measure of the relative accuracy with which the nonlinear functions can be computed [Default: 0e0]
 |
 |  FEASPB
 |      Corresponds to 'Feasible point' in snOptA, which specif

After setup, the NLP model can be solved to local optimality by passing an initial guess for the decision variables and the parameters to the method `solve`:

In [5]:
NLP.setup()
NLP.solve( [0.,0.], [3.] )

 
 SNMEMA EXIT 100 -- finished successfully
 SNMEMA INFO 104 -- memory requirements estimated


1

 

 Nonlinear constraints       1     Linear constraints       1
 Nonlinear variables         2     Linear variables         0
 Jacobian  variables         2     Objective variables      0
 Total constraints           2     Total variables          2
 

 
 The user has defined       2   out of       2   first  derivatives
 

 Major Minors     Step   nCon Feasible  Optimal  MeritFunction     nS Penalty
     0      2               1 (0.0E+00) 1.0E+00 -0.0000000E+00      2           r
     1      1  1.0E+00      2 (0.0E+00) 1.0E+00  2.0000000E+00      2           rl
     2      1  1.0E+00      4  5.0E-01  2.5E-02  4.0000000E+00      1         sm l
     3      1  5.6E-02      6  4.8E-01  6.8E-03  3.4907908E+00      1 1.0E+00
     4      1  7.7E-02      8  4.4E-01  1.5E-02  3.4911024E+00      1 1.0E+00
     5      1  3.7E-02     10  4.3E-01  1.8E-02  3.4875518E+00      1 1.0E+00
     6      1  4.8E-02     12  4.1E-01  2.1E-02  3.4849171E+00      1 1.0E+00
     7      1  6.2E-02     14  3.9E

In [6]:
print( "status:", NLP.status )
print( "solution point:", NLP.solution.x )
print( "solution value:", NLP.solution.f[0] )
#print( NLP.solution )

status: STATUS.SUCCESSFUL
solution point: [1.7320508075691967, 1.7320508075692664]
solution value: 3.4641016151384636


Similarly, multistart can be used to solve the NLP model by passing the number of sampled initial points - this is only possible when the domain of the deicsion variables is bounded:

In [7]:
NLP.options.DISPLEVEL = 0;
NLP.options.MAXTHREAD = 4;
NLP.solve( 8, [3.] )

print( "status:", NLP.status )
print( "solution point:", NLP.solution.x )
print( "solution value:", NLP.solution.f[0] )
#print( NLP.solution )

status: STATUS.SUCCESSFUL
solution point: [6.0, 0.5]
solution value: 6.5
